<a href="https://colab.research.google.com/github/drfperez/OMRSuite/blob/main/OMRSuite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ==============================================================================
# SUITE INTEGRAL OMR (GOOGLE COLAB VERSION)
# Execució en Python nadiu + OpenCV + Pandas
# ==============================================================================

import cv2
import numpy as np
import pandas as pd
import random
import os
import io
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

# --- CONSTANTS DE GEOMETRIA (IDENTIQUES AL TEU CODI WEB) ---
TEMPLATE_W = 1050
TEMPLATE_H = 1485
CIRCLE_RADIUS = 8
ROI_SIZE = 10
OPTIONS = ['A', 'B', 'C', 'D']

X_START_ID = 330
Y_DESENES = 160
Y_UNITATS = 205
ID_STEP_X = 35

X_COLS_DIRECTES = [
    [150, 190, 230, 270],
    [450, 490, 530, 570],
    [750, 790, 830, 870]
]

# ==============================================================================
# 1. MOTOR DE DIBUIX DE PLANTILLES (PIL)
# ==============================================================================
def draw_template_pil(title_text="FULL DE RESPOSTES", nom="", id_alumne="", assignatura="",
                      data_text="", grup_text="", respostes_array=None, num_questions=150, include_signature=False):

    # Imprimit a doble resolució per a màxima qualitat
    scale_print = 2
    w, h = TEMPLATE_W * scale_print, TEMPLATE_H * scale_print
    img = Image.new('RGB', (w, h), color='white')
    draw = ImageDraw.Draw(img)

    try:
        font_large = ImageFont.truetype("DejaVuSans-Bold.ttf", 28 * scale_print)
        font_main = ImageFont.truetype("DejaVuSans.ttf", 20 * scale_print)
        font_small = ImageFont.truetype("DejaVuSans.ttf", 14 * scale_print)
    except:
        font_large = font_main = font_small = ImageFont.load_default()

    def s(val): return int(val * scale_print)

    # Capçalera
    draw.text((s(60), s(40)), f"Nom: {nom}", fill='black', font=font_main)
    draw.text((s(440), s(40)), f"Assignatura: {assignatura}", fill='black', font=font_main)
    draw.text((s(740), s(40)), f"Grup: {grup_text}", fill='black', font=font_main)
    draw.text((s(860), s(40)), f"Data: {data_text}", fill='black', font=font_main)

    # Títol
    draw.text((s(60), s(95)), title_text, fill='black', font=font_large)
    draw.line([(s(60), s(130)), (s(990), s(130))], fill='black', width=s(2))

    # Fites de registre (4 cantonades)
    marker_s = s(40)
    for cx, cy in [(60, 160), (990, 160), (60, 1420), (990, 1420)]:
        draw.rectangle([s(cx)-marker_s//2, s(cy)-marker_s//2, s(cx)+marker_s//2, s(cy)+marker_s//2], fill='black')

    # ID Alumne
    draw.text((s(120), s(Y_DESENES)), "ID Alumne - Desenes:", fill='black', font=font_small)
    draw.text((s(120), s(Y_UNITATS)), "ID Alumne - Unitats:", fill='black', font=font_small)

    id_str = str(id_alumne).zfill(2) if id_alumne != "" else ""
    tens = int(id_str[0]) if len(id_str) >= 2 and id_str[0].isdigit() else -1
    units = int(id_str[1]) if len(id_str) >= 2 and id_str[1].isdigit() else -1

    for i in range(10):
        # Desenes
        cx, cy = s(X_START_ID + i * ID_STEP_X + 15), s(Y_DESENES + 7.5)
        r = s(CIRCLE_RADIUS)
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline='black', width=s(1.5))
        draw.text((cx - s(4), cy - s(18)), str(i), fill='black', font=font_small)
        if tens == i: draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill='black')

        # Unitats
        cy_u = s(Y_UNITATS + 7.5)
        draw.ellipse([cx-r, cy_u-r, cx+r, cy_u+r], outline='black', width=s(1.5))
        draw.text((cx - s(4), cy_u - s(18)), str(i), fill='black', font=font_small)
        if units == i: draw.ellipse([cx-r, cy_u-r, cx+r, cy_u+r], fill='black')

    # Columnes de preguntes
    num_cols = 1 if num_questions <= 50 else (2 if num_questions <= 100 else 3)
    count = 0

    for col in range(num_cols):
        for c in range(4):
            draw.text((s(X_COLS_DIRECTES[col][c] + 10), s(230)), OPTIONS[c], fill='black', font=font_small)

        for i in range(50):
            count += 1
            if count > num_questions: break

            y = int((53 + i * (225 / 49)) * 5)
            draw.text((s(X_COLS_DIRECTES[col][0] - 35), s(y)), str(count), fill='black', font=font_small)

            for c in range(4):
                cx, cy = s(X_COLS_DIRECTES[col][c] + 15), s(y + 7.5)
                r = s(CIRCLE_RADIUS)
                draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline='black', width=s(1.5))

                if respostes_array and count <= len(respostes_array):
                    if respostes_array[count-1] == OPTIONS[c]:
                        draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill='black')

    # Quadre de Signatura
    if include_signature:
        draw.text((s(700), s(140)), "Signatura / Signature:", fill='black', font=font_small)
        draw.rectangle([s(700), s(155), s(940), s(220)], outline='black', width=s(1.5))

    return img

# ==============================================================================
# 2. MOTOR DE CORRECCIÓ AUTOMÀTICA (OPENCV PYTHON NADIU)
# ==============================================================================
def redrecar_imatge_opencv(cv_img):
    """Alinea l'examen buscant les 4 fites negres de les cantonades"""
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    markers = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 200 < area < 30000:
            x, y, w, h = cv2.boundingRect(cnt)
            aspect = float(w) / h
            if 0.7 <= aspect <= 1.3:
                hull = cv2.convexHull(cnt)
                if (area / cv2.contourArea(hull)) > 0.8:
                    M = cv2.moments(cnt)
                    if M["m00"] != 0:
                        markers.append({'x': M["m10"]/M["m00"], 'y': M["m01"]/M["m00"], 'area': area})

    markers.sort(key=lambda m: m['area'], reverse=True)
    top_m = markers[:4]

    if len(top_m) == 4:
        top_m.sort(key=lambda m: (m['x'] + m['y']))
        tl, br = top_m[0], top_m[3]
        rem = sorted([top_m[1], top_m[2]], key=lambda m: (m['x'] - m['y']))
        bl, tr = rem[0], rem[1]

        src_pts = np.array([[tl['x'], tl['y']], [tr['x'], tr['y']], [bl['x'], bl['y']], [br['x'], br['y']]], dtype="float32")
        dst_pts = np.array([[60, 160], [TEMPLATE_W - 60, 160], [60, 1420], [TEMPLATE_W - 60, 1420]], dtype="float32")

        M = cv2.getPerspectiveTransform(src_pts, dst_pts)
        aligned = cv2.warpPerspective(cv_img, M, (TEMPLATE_W, TEMPLATE_H))
        return aligned
    else:
        # Redimensionat d'emergència si les cantonades estan tapades
        return cv2.resize(cv_img, (TEMPLATE_W, TEMPLATE_H), interpolation=cv2.INTER_AREA)

def es_marca_obscura(gray_img, cx, cy):
    """Analitza si la bombolla està pintada avaluant la intensitat dels píxels"""
    half_r = ROI_SIZE // 2
    roi = gray_img[int(cy - half_r):int(cy + half_r), int(cx - half_r):int(cx + half_r)]
    if roi.size == 0: return False
    pings = np.sum(roi < 120)
    return (pings / roi.size) > 0.40

def llegir_dades_examen(aligned_img, max_q):
    """Llegeix l'ID de l'alumne i les respostes de les bombolles"""
    gray = cv2.cvtColor(aligned_img, cv2.COLOR_BGR2GRAY)

    # ID Alumne
    d, u = -1, -1
    for i in range(10):
        if es_marca_obscura(gray, X_START_ID + i * ID_STEP_X + 15, Y_DESENES + 7.5): d = i
        if es_marca_obscura(gray, X_START_ID + i * ID_STEP_X + 15, Y_UNITATS + 7.5): u = i

    id_alumne = f"{d}{u}" if (d != -1 and u != -1) else "NO LLEGIT"

    # Preguntes
    respostes = []
    num_cols = 1 if max_q <= 50 else (2 if max_q <= 100 else 3)
    count = 0

    for col in range(num_cols):
        for i in range(50):
            if count >= max_q: break
            cy = int((53 + i * (225 / 49)) * 5) + 7.5
            marcades = []

            for c in range(4):
                cx = X_COLS_DIRECTES[col][c] + 15
                if es_marca_obscura(gray, cx, cy): marcades.append(OPTIONS[c])

            if len(marcades) == 1: respostes.append(marcades[0])
            elif len(marcades) > 1: respostes.append('MULTIPLE')
            else: respostes.append('')

            count += 1

    return id_alumne, respostes

# ==============================================================================
# 3. INTERFÍCIE D'USUARI INTERACTIVA A COLAB
# ==============================================================================
def menu_principal():
    print("==================================================")
    print("      SUITE OMR INTEGRAL - GOOGLE COLAB          ")
    print("==================================================")
    print("1. Generar Plantilla Buida (JPG)")
    print("2. Generar Pauta de Correcció Aleatòria (JPG)")
    print("3. Generar Exàmens Personalitzats per Llista (PDF)")
    print("4. CORREGIR EXÀMENS EN LOTS (Pauta + Exàmens)")
    print("==================================================")

    opcio = input("Tria una opció (1-4): ").strip()

    if opcio == '1':
        num_q = int(input("Nombre de preguntes (Ex: 150): ") or 150)
        img = draw_template_pil(title_text="FULL DE RESPOSTES", num_questions=num_q)
        img.save("Plantilla_Buida.jpg")
        files.download("Plantilla_Buida.jpg")
        print("✅ Plantilla descarregada!")

    elif opcio == '2':
        num_q = int(input("Nombre de preguntes (Ex: 150): ") or 150)
        pauta = [random.choice(OPTIONS) for _ in range(num_q)]
        img = draw_template_pil(title_text="PAUTA PROFESSOR", respostes_array=pauta, num_questions=num_q)
        img.save("Pauta_Professor.jpg")
        files.download("Pauta_Professor.jpg")
        print(f"✅ Pauta generada: {''.join(pauta)}")

    elif opcio == '3':
        assignatura = input("Assignatura: ")
        grup = input("Grup: ")
        data_text = input("Data: ")
        num_q = int(input("Nombre de preguntes (Ex: 50): ") or 50)

        print("\nEnganxa la llista d'alumnes (Format: Nom, ID):")
        print("(Prem Enter dues vegades quan acabis)")
        lines = []
        while True:
            line = input()
            if not line: break
            lines.append(line)

        pages = []
        for line in lines:
            parts = line.split(',')
            nom = parts[0].strip()
            id_a = parts[1].strip() if len(parts) > 1 else ""
            img = draw_template_pil(title_text="FULL DE RESPOSTES", nom=nom, id_alumne=id_a,
                                    assignatura=assignatura, grup_text=grup, data_text=data_text,
                                    num_questions=num_q, include_signature=True)
            pages.append(img.convert('RGB'))

        if pages:
            pdf_path = f"Examens_{assignatura if assignatura else 'Personalitzats'}.pdf"
            pages[0].save(pdf_path, save_all=True, append_images=pages[1:])
            files.download(pdf_path)
            print("✅ PDF d'exàmens generat i descarregat!")

    elif opcio == '4':
        print("\n--- 1. PUJA LA IMATGE DE LA PAUTA (PROFESSOR) ---")
        uploaded_pauta = files.upload()
        pauta_filename = list(uploaded_pauta.keys())[0]

        print("\n--- 2. PUJA ELS EXÀMENS DELS ALUMNES (SELECCIONA TOTS) ---")
        uploaded_examens = files.upload()

        num_q = int(input("\nNombre de preguntes a avaluar (Ex: 150): ") or 150)
        penalitzacio = float(input("Penalització per error (Ex: 0.25): ") or 0.25)

        # 1. Processar Pauta
        pauta_cv = cv2.imread(pauta_filename)
        pauta_aligned = redrecar_imatge_opencv(pauta_cv)
        _, pauta_respostes = llegir_dades_examen(pauta_aligned, num_q)

        # 2. Processar Exàmens d'Alumnes
        resultats = []

        print("\nProcessant exàmens...")
        for fname in uploaded_examens.keys():
            if fname == pauta_filename: continue

            exam_cv = cv2.imread(fname)
            aligned = redrecar_imatge_opencv(exam_cv)
            id_alumne, respostes = llegir_dades_examen(aligned, num_q)

            ok, err, bl = 0, 0, 0
            for j in range(num_q):
                if respostes[j] == '': bl += 1
                elif respostes[j] == pauta_respostes[j]: ok += 1
                else: err += 1

            nota_final = max(0, ((ok - (err * penalitzacio)) / num_q) * 10)

            resultats.append({
                'Fitxer': fname,
                'ID Alumne': id_alumne,
                'Encerts': ok,
                'Errors': err,
                'Blancs': bl,
                'Nota Final': round(nota_final, 2)
            })

        # Mostrar Resultats en una Taula Neta
        df = pd.DataFrame(resultats)
        print("\n================ RESULTATS DE LA CORRECCIÓ ================\n")
        print(df.to_string(index=False))

        # Guardar en CSV i descarregar
        df.to_csv("Resultats_OMR_Colab.csv", index=False)
        files.download("Resultats_OMR_Colab.csv")
        print("\n✅ Fitxer 'Resultats_OMR_Colab.csv' descarregat amb èxit!")

# Executar el menú
menu_principal()

      SUITE OMR INTEGRAL - GOOGLE COLAB          
1. Generar Plantilla Buida (JPG)
2. Generar Pauta de Correcció Aleatòria (JPG)
3. Generar Exàmens Personalitzats per Llista (PDF)
4. CORREGIR EXÀMENS EN LOTS (Pauta + Exàmens)
Tria una opció (1-4): 4

--- 1. PUJA LA IMATGE DE LA PAUTA (PROFESSOR) ---


Saving Pauta (5).jpg to Pauta (5).jpg

--- 2. PUJA ELS EXÀMENS DELS ALUMNES (SELECCIONA TOTS) ---


Saving Alumne_74.jpg to Alumne_74.jpg
Saving Alumne_26 (1).jpg to Alumne_26 (1).jpg
Saving Alumne_78 (1).jpg to Alumne_78 (1).jpg
Saving Alumne_33.jpg to Alumne_33.jpg
Saving Alumne_79.jpg to Alumne_79.jpg
Saving Alumne_46 (1).jpg to Alumne_46 (1).jpg
Saving Alumne_78.jpg to Alumne_78.jpg
Saving Alumne_07.jpg to Alumne_07.jpg
Saving Alumne_85.jpg to Alumne_85.jpg
Saving Alumne_36.jpg to Alumne_36.jpg
Saving Alumne_86.jpg to Alumne_86.jpg
Saving Alumne_05.jpg to Alumne_05.jpg
Saving Alumne_21.jpg to Alumne_21.jpg
Saving Alumne_48.jpg to Alumne_48.jpg
Saving Alumne_17 (1).jpg to Alumne_17 (1).jpg
Saving Alumne_09.jpg to Alumne_09.jpg
Saving Alumne_88.jpg to Alumne_88.jpg
Saving Alumne_17.jpg to Alumne_17.jpg
Saving Alumne_53.jpg to Alumne_53.jpg
Saving Alumne_46.jpg to Alumne_46.jpg

Nombre de preguntes a avaluar (Ex: 150): 150
Penalització per error (Ex: 0.25): 0.25

Processant exàmens...

================ RESULTATS DE LA CORRECCIÓ ================

           Fitxer ID Alumne  Encerts 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Fitxer 'Resultats_OMR_Colab.csv' descarregat amb èxit!
